# Methodology and Validation

This notebook walks through the same core workflow used by the Streamlit app: load historical prices, calculate returns, build the default portfolio, run both simulation engines, and compare the resulting risk metrics.

The notebook is meant to make the assumptions inspectable. It is not a prediction notebook.

## Setup

Default portfolio:

- SPY: 40%
- QQQ: 25%
- TLT: 25%
- GLD: 10%

Default simulation settings are a $10,000 starting value, 5-year lookback, 10,000 simulations, 252 trading-day horizon, seed 42, and benchmark SPY.

In [ ]:
import pandas as pd

from monte_carlo_portfolio_lab.config import DEFAULT_CONFIG, DEFAULT_PORTFOLIO, SCENARIOS
from monte_carlo_portfolio_lab.data import YFinanceDataProvider
from monte_carlo_portfolio_lab.metrics import summarize_simulation
from monte_carlo_portfolio_lab.portfolio import Portfolio
from monte_carlo_portfolio_lab.returns import calculate_log_returns, correlation_matrix
from monte_carlo_portfolio_lab.simulation import HistoricalBootstrapEngine, ParametricMonteCarloEngine, SimulationConfig


## Load Historical Data

The app uses adjusted close prices from Yahoo Finance via `yfinance`. Missing values are forward-filled and rows with incomplete data are dropped before returns are calculated.

In [ ]:
portfolio = Portfolio.from_mapping(DEFAULT_PORTFOLIO, DEFAULT_CONFIG.initial_capital)
benchmark = Portfolio.from_mapping({DEFAULT_CONFIG.benchmark: 1.0}, DEFAULT_CONFIG.initial_capital)
tickers = sorted(set(portfolio.tickers + [DEFAULT_CONFIG.benchmark]))

prices = YFinanceDataProvider().get_price_history(tickers, DEFAULT_CONFIG.lookback_years)
returns = calculate_log_returns(prices)

prices.tail(), returns.describe()


## Asset Relationships

Correlations matter because the parametric engine uses the historical covariance matrix and the bootstrap engine resamples full return rows, preserving observed same-day co-movement.

In [ ]:
correlation_matrix(returns)


## Run the Two Engines

The Parametric Monte Carlo engine samples from a multivariate normal distribution using historical mean returns and covariance. The Historical Bootstrap engine resamples observed historical return rows.

In [ ]:
def make_config(scenario_name: str = "Normal Market") -> SimulationConfig:
    scenario = SCENARIOS[scenario_name]
    return SimulationConfig(
        horizon_days=DEFAULT_CONFIG.horizon_days,
        simulations=DEFAULT_CONFIG.simulations,
        seed=DEFAULT_CONFIG.seed,
        volatility_multiplier=scenario["volatility_multiplier"],
        mean_shift=scenario["mean_shift"],
        correlation_stress=scenario["correlation_stress"],
    )


config = make_config()
parametric_result = ParametricMonteCarloEngine().run(returns, portfolio, config)
bootstrap_result = HistoricalBootstrapEngine().run(returns, portfolio, config)

parametric_benchmark = ParametricMonteCarloEngine().run(returns, benchmark, config)
bootstrap_benchmark = HistoricalBootstrapEngine().run(returns, benchmark, config)


## Compare Metrics

Metrics are calculated from simulated paths. Terminal value, VaR, and CVaR are distribution metrics across terminal outcomes. Volatility, Sharpe, Sortino, and drawdown are calculated path by path first and then summarized.

In [ ]:
parametric_metrics = summarize_simulation(
    parametric_result,
    DEFAULT_CONFIG.target_return,
    benchmark_result=parametric_benchmark,
)
bootstrap_metrics = summarize_simulation(
    bootstrap_result,
    DEFAULT_CONFIG.target_return,
    benchmark_result=bootstrap_benchmark,
)

summary_rows = pd.DataFrame(
    [parametric_metrics, bootstrap_metrics],
    index=["Parametric Monte Carlo", "Historical Bootstrap"],
)
summary_rows[
    [
        "median_terminal_value",
        "expected_terminal_value",
        "p05_terminal_value",
        "p95_terminal_value",
        "probability_of_loss",
        "probability_target_return",
        "probability_outperforming_benchmark_median",
        "median_maximum_drawdown",
        "value_at_risk_95",
        "conditional_value_at_risk_95",
    ]
]


## Stress Test

The Market Stress Test increases volatility, applies a negative annual mean-return shock, and pushes correlations closer to 1.0. This is not a forecast; it is a sensitivity check.

In [ ]:
stress_config = make_config("Market Stress Test")
stress_result = ParametricMonteCarloEngine().run(returns, portfolio, stress_config)
stress_benchmark = ParametricMonteCarloEngine().run(returns, benchmark, stress_config)
stress_metrics = summarize_simulation(
    stress_result,
    DEFAULT_CONFIG.target_return,
    benchmark_result=stress_benchmark,
)

pd.Series(stress_metrics)[
    [
        "median_terminal_value",
        "p05_terminal_value",
        "p95_terminal_value",
        "probability_of_loss",
        "median_maximum_drawdown",
        "conditional_value_at_risk_95",
    ]
]


## Limitations

- Historical returns do not predict future returns.
- The parametric engine assumes normally distributed daily log returns.
- The bootstrap engine is limited to patterns observed in the lookback window.
- Tail events can exceed the modeled range.
- Benchmark comparison is distribution-based and should not be read as a recommendation.